# Mapa de influencia y desinformación en la conversación sobre el NFL 2014
> - **Equipo:**
>   - Chan Martin Luis Gael
>   - Euan Madero Rosy Guadalupe
>   - Hernández Bermúdez Adrián
>   - Ojeda Novelo Dominique Salem
>   - Saucedo Ramírez Shaiel Eduardo
>   - Vargas Botello Dania Belén
> - **Fecha:** 14 de julio de 2026
> - **Institución:** Universidad Politécnica de Quintana Roo
> - **Asignatura:** Minería de Datos
> - **Carrera:** Ing. en Software
> - **Profesor:** Victor Tuz Contreras

### 1. Contexto del Proyecto
- Tema: Mapa de influencia y desinformación en la conversación sobre el NFL 2014
- Enfoque: Análisis de redes sociales (Twitter/X) usando minería de datos y análisis de grafos
- Temporada: NFL 2014 (Super Bowl XLIX: Patriots vs Seahawks) (co-anfitriones del NFL 2014)
- Alcance: 75 nodos (cuentas) y 108 aristas (menciones) en un grafo dirigido ponderado
### 2. Propósito
- Identificar quién tiene más influencia en la conversación sobre el NFL 2014
- Detectar comunidades naturales dentro de la red de interacciones
- Encontrar y analizar bots que manipulan la conversación
- Entender cómo fluye la información y la desinformación a través de la red
### 3. Características del Proyecto
- Pipeline de 5 tareas: construcción del grafo → detección de comunidades (Louvain) → métricas de centralidad → detección de bots → visualización interactiva
- 7 tipos de cuentas: fans, periodistas, medios, equipos oficiales, hashtags, influencers y bots
- Tecnologías: Python (pandas, NetworkX, python-louvain, PyVis)
- Visualización interactiva: grafo HTML con filtros, estadísticas y colores por comunidad
- Reproducibilidad: semilla fija (random_state=42)
### 4. Ventajas / Fortalezas
- Detección de bots de dos niveles: bots confirmados (etiquetados) + candidatos por reglas estructurales (reduce falsos positivos)
- Interpretación cualitativa de comunidades: cruza tipo de cuenta y equipo para nombrar cada comunidad automáticamente
- Visualización autónoma: el HTML generado funciona offline con librerías JS empaquetadas
- Manejo cuidadoso de pesos: invierte pesos para betweenness (NetworkX usa pesos como distancia) y los usa directamente para eigenvector
- Grafo sparse pero informativo: modularidad de 0.6070 indica estructura comunitaria fuerte
### 5. Cómo ayuda a resolver el problema de los bots
- Identificación: 5 bots confirmados detectados + 3 candidatos adicionales por anomalias estructurales (coeficiente de agrupamiento = 0, conexión a 3+ comunidades)
- Impacto medido: bot_fut_01 alcanza el puesto 7 en influencia (eigenvector centrality), demostrando que los bots logran influencia estructural significativa
- Patrón revelador: todos los bots son "solo fuente" — mencionan a otros pero nadie los menciona a ellos (patrón típico de amplificación artificial)
- Transparencia: la whitelist evita que cuentas oficiales sean flagged erróneamente
- Visualización: el botón "Show only bots" permite a los analistas ver exactamente dónde están los bots en la red

Este notebook documenta, el script `nfl2014_analisis_red.py`.
El código original se reorganiza en 6 secciones temáticas para facilitar su lectura y ejecución
paso a paso:

1. **Construcción del grafo de menciones** entre cuentas (fans, equipos, medios, bots, influencers).
2. **Detección de comunidades temáticas** (por equipo, tipo de cuenta, idioma) con Louvain.
3. **Métricas de centralidad** y top de influencers globales.
4. **Detección de bots** y su impacto en la estructura de la conversación.
5. **Dashboard / reporte visual** de los hallazgos.
6. **Código main** — orquestación de punta a punta (CLI, generación de outputs).


**Archivos de entrada esperados** (mismo directorio, o rutas pasadas por CLI):

| Archivo     | Columnas requeridas         |
|-------------|------------------------------|
| `edges.csv` | `source`, `target`, `weight` |
| `nodes.csv` | `node`, `tipo`, `equipo`       |


## 0. Imports y carga de datos


In [ ]:
import sys
import argparse

import pandas as pd
import networkx as nx

try:
    import community as community_louvain  # python-louvain
except ImportError:
    sys.exit(
        "Falta la dependencia 'python-louvain'.\n"
        "Instálala con: pip install python-louvain"
    )

from centralidad import (
    calcular_centralidades,
    reportar_centralidades,
    detectar_bots,
    generar_visualizacion,
)

In [ ]:
def cargar_datos(edges_path: str, nodes_path: str):
    try:
        edges_df = pd.read_csv(edges_path)
        nodes_df = pd.read_csv(nodes_path)
    except FileNotFoundError as e:
        sys.exit(f"No se encontró el archivo: {e.filename}")

    columnas_edges_esperadas = {"source", "target", "weight"}
    columnas_nodes_esperadas = {"node", "tipo", "equipo"}

    faltantes_edges = columnas_edges_esperadas - set(edges_df.columns)
    faltantes_nodes = columnas_nodes_esperadas - set(nodes_df.columns)

    if faltantes_edges:
        sys.exit(f"edges.csv no contiene las columnas requeridas: {faltantes_edges}")
    if faltantes_nodes:
        sys.exit(f"nodes.csv no contiene las columnas requeridas: {faltantes_nodes}")

    return edges_df, nodes_df

## 1. Construcción del grafo de menciones

**Objetivo:** construir un grafo de menciones entre cuentas (fans, equipos, medios,
bots, influencers) a partir de `edges.csv`.

- `construir_grafo`: crea un `nx.DiGraph` a partir del edgelist (`source` → `target`),
  conservando `weight` como atributo de arista. Es un grafo **dirigido**, porque una
  mención tiene dirección (quién menciona a quién).
- `reportar_grafo`: imprime estadísticas descriptivas del grafo — número de nodos,
  número de aristas, densidad, y el top 5 de cuentas por grado ponderado (volumen de
  interacción total, sumando menciones entrantes y salientes).


In [ ]:
def construir_grafo(edges_df: pd.DataFrame) -> nx.DiGraph:
    """Construye un grafo dirigido: las menciones tienen dirección (quién menciona a quién)."""
    G = nx.from_pandas_edgelist(
        edges_df,
        source="source",
        target="target",
        edge_attr="weight",
        create_using=nx.DiGraph(),
    )
    return G

In [ ]:
def reportar_grafo(G: nx.DiGraph) -> None:
    print("=" * 70)
    print("TAREA 1 — CONSTRUCCIÓN DEL GRAFO")
    print("=" * 70)
    print(f"Número de nodos : {G.number_of_nodes()}")
    print(f"Número de enlaces (aristas): {G.number_of_edges()}")
    print(f"Densidad del grafo: {nx.density(G):.6f}")

    if G.number_of_nodes() > 0:
        grados = dict(G.degree(weight="weight"))
        top5 = sorted(grados.items(), key=lambda x: x[1], reverse=True)[:5]
        print("\nTop 5 cuentas por grado ponderado (volumen de interacción):")
        for nodo, grado in top5:
            print(f"  {nodo}: {grado}")
    print()

### Ejecución de la Sección 1

Carga `edges.csv` / `nodes.csv`, construye el grafo dirigido y muestra el reporte.


In [ ]:
edges_df, nodes_df = cargar_datos("edges.csv", "nodes.csv")

G = construir_grafo(edges_df)
reportar_grafo(G)

TAREA 1 — CONSTRUCCIÓN DEL GRAFO
Número de nodos : 94
Número de enlaces (aristas): 132
Densidad del grafo: 0.037386

Top 5 cuentas por grado ponderado (volumen de interacción):
  seleccion_mexico: 18
  seleccion_canada: 17
  seleccion_usa: 16
  espn_usa: 7
  bbc_sports: 7



## 2. Detección de comunidades temáticas (Louvain)

**Objetivo:** detectar comunidades temáticas (por equipo, tipo de cuenta, idioma)
dentro de la red de menciones.

- `detectar_comunidades`: convierte el grafo dirigido a no dirigido (Louvain está
  definido sobre grafos no dirigidos) y aplica `community_louvain.best_partition`
  con `weight="weight"` y `random_state=42` para resultados reproducibles. Calcula
  también la **modularidad global** de la partición encontrada.
- `interpretar_comunidades`: cruza cada comunidad con los metadatos de `nodes.csv`
  (`tipo`, `equipo`) para darle una lectura cualitativa: qué equipo domina cada
  comunidad, si hay concentración sospechosa de bots (>30%), si es una comunidad
  de medios/cobertura global (conecta varios equipos), o si hay presencia relevante
  de influencers marcando agenda. Devuelve un `DataFrame` resumen.


In [ ]:
def detectar_comunidades(G: nx.DiGraph):
    print("=" * 70)
    print("TAREA 2 — DETECCIÓN DE COMUNIDADES (LOUVAIN)")
    print("=" * 70)

    # Louvain está definido para grafos no dirigidos; convertimos preservando pesos.
    G_und = G.to_undirected()

    particion = community_louvain.best_partition(G_und, weight="weight", random_state=42)
    modularidad = community_louvain.modularity(particion, G_und, weight="weight")

    comunidades = {}
    for nodo, com_id in particion.items():
        comunidades.setdefault(com_id, []).append(nodo)

    print(f"Modularidad global: {modularidad:.4f}")
    print(f"Número de comunidades detectadas: {len(comunidades)}\n")

    print("Tamaño de cada comunidad:")
    for com_id, miembros in sorted(comunidades.items(), key=lambda x: -len(x[1])):
        print(f"  Comunidad {com_id}: {len(miembros)} nodos")
    print()

    return particion, comunidades, modularidad

In [ ]:
def interpretar_comunidades(comunidades: dict, nodes_df: pd.DataFrame) -> pd.DataFrame:
    """
    Cruza cada comunidad detectada con los metadatos de nodes.csv (tipo, equipo)
    para dar una lectura cualitativa: ¿es una comunidad de México, USA, Canadá,
    medios globales?, y si hay concentración sospechosa de bots.
    """
    print("=" * 70)
    print("INTERPRETACIÓN DE COMUNIDADES")
    print("=" * 70)

    nodes_idx = nodes_df.set_index("node")
    resumen = []

    for com_id, miembros in sorted(comunidades.items(), key=lambda x: -len(x[1])):
        info = nodes_idx.reindex(miembros)

        equipos = info["equipo"].value_counts(dropna=True)
        tipos = info["tipo"].value_counts(dropna=True)

        equipo_dominante = equipos.idxmax() if not equipos.empty else "N/D"
        pct_equipo = (equipos.max() / len(miembros) * 100) if not equipos.empty else 0.0
        pct_bots = (tipos.get("bot", 0) / len(miembros) * 100) if len(miembros) else 0.0

        print(f"\n--- Comunidad {com_id} ({len(miembros)} nodos) ---")
        print(f"Equipo dominante: {equipo_dominante} ({pct_equipo:.1f}% de la comunidad)")

        print("Distribución por equipo:")
        for equipo, cnt in equipos.items():
            print(f"    {equipo}: {cnt}")

        print("Distribución por tipo de cuenta:")
        for tipo, cnt in tipos.items():
            print(f"    {tipo}: {cnt}")

        etiqueta = f"Comunidad dominada por {equipo_dominante}"
        if pct_bots > 30:
            etiqueta += f" — ALERTA: {pct_bots:.1f}% de bots, posible amplificación artificial"
        elif tipos.get("medio", 0) >= 2 and len(equipos) > 2:
            etiqueta = "Comunidad de medios/cobertura global (conecta varios equipos)"
        elif tipos.get("influencer", 0) >= 1 and tipos.get("influencer", 0) / len(miembros) > 0.1:
            etiqueta += " — presencia relevante de influencers que marcan agenda"

        print(f"Interpretación: {etiqueta}")

        resumen.append({
            "comunidad": com_id,
            "tamano": len(miembros),
            "equipo_dominante": equipo_dominante,
            "pct_equipo_dominante": round(pct_equipo, 1),
            "pct_bots": round(pct_bots, 1),
            "interpretacion": etiqueta,
        })

    return pd.DataFrame(resumen)

### Ejecución de la Sección 2

Usa el grafo `G` construido en la Sección 1.


In [ ]:
particion, comunidades, modularidad = detectar_comunidades(G)
resumen_df = interpretar_comunidades(comunidades, nodes_df)
resumen_df

TAREA 2 — DETECCIÓN DE COMUNIDADES (LOUVAIN)
Modularidad global: 0.5442
Número de comunidades detectadas: 8

Tamaño de cada comunidad:
  Comunidad 1: 11 nodos
  Comunidad 3: 10 nodos
  Comunidad 9: 2 nodos
  Comunidad 2: 7 nodos
  Comunidad 6: 6 nodos
  Comunidad 7: 4 nodos
  Comunidad 8: 2 nodos
  Comunidad 9: 2 nodos

INTERPRETACIÓN DE COMUNIDADES

--- Comunidad 5 (16 nodos) ---
País dominante: mexico (68.8% de la comunidad)
Distribución por país:
    mexico: 11
    global: 4
    canada: 1
Distribución por tipo de cuenta:
    periodista: 5
    fan: 4
    medio: 3
    seleccion: 1
    influencer: 1
    bot: 1
    hashtag: 1
Interpretación: Comunidad de medios/cobertura global (conecta varios países)

--- Comunidad 1 (16 nodos) ---
País dominante: usa (68.8% de la comunidad)
Distribución por país:
    usa: 11
    global: 3
    canada: 2
Distribución por tipo de cuenta:
    fan: 5
    periodista: 5
    medio: 3
    seleccion: 1
    influencer: 1
    bot: 1
Interpretación: Comunidad de m

,comunidad,tamano,equipo_dominante,pct_equipo_dominante,pct_bots,interpretacion
0,5,16,mexico,68.8,6.2,Comunidad de medios/cobertura global (conecta ...
1,1,16,usa,68.8,6.2,Comunidad de medios/cobertura global (conecta ...
2,2,15,canada,80.0,6.7,Comunidad dominada por canada
3,6,13,canada,46.2,7.7,Comunidad dominada por canada
4,3,9,mexico,88.9,0.0,Comunidad dominada por mexico
5,7,9,usa,55.6,0.0,Comunidad dominada por usa
6,0,8,mexico,100.0,0.0,Comunidad dominada por mexico
7,4,8,mexico,25.0,12.5,Comunidad dominada por mexico


## 3. Métricas de centralidad y top influencers globales

**Objetivo:** calcular métricas de centralidad sobre el grafo y encontrar el top de
influencers globales.

Esta sección se apoya en dos funciones **importadas desde el módulo externo
`centralidad.py`**:

- `calcular_centralidades(G)`: calcula las métricas de centralidad sobre el grafo
  `G` (p. ej. grado, intermediación, cercanía, eigenvector u otras, según la
  implementación del módulo) y devuelve un `DataFrame` con los resultados por nodo.
- `reportar_centralidades(centralidad_df, nodes_df, particion, top_n=10)`: cruza
  las centralidades con los metadatos de `nodes.csv` y la partición de comunidades
  (Sección 2) para construir el ranking de **top influencers** (por defecto, los
  10 primeros), enriquecido con tipo de cuenta, equipo y comunidad a la que pertenecen.

In [ ]:
centralidad_df = calcular_centralidades(G)
top_influencers_df = reportar_centralidades(centralidad_df, nodes_df, particion, top_n=10)
top_influencers_df

TAREA 3 — CENTRALIDAD E INFLUENCERS

Top 10 cuentas más influyentes (por eigenvector centrality):

  seleccion_usa  [seleccion · usa · comunidad 1]
    degree=0.1720  betweenness=0.2517  eigenvector=0.3827
    Justificación: alto volumen de interacción directa, actúa como puente entre comunidades, conectada a otras cuentas muy influyentes
  seleccion_mexico  [seleccion · mexico · comunidad 5]
    degree=0.1935  betweenness=0.3077  eigenvector=0.3798
    Justificación: alto volumen de interacción directa, actúa como puente entre comunidades, conectada a otras cuentas muy influyentes
  seleccion_canada  [seleccion · canada · comunidad 2]
    degree=0.1828  betweenness=0.3325  eigenvector=0.3609
    Justificación: alto volumen de interacción directa, actúa como puente entre comunidades, conectada a otras cuentas muy influyentes
  bbc_sports  [medio · global · comunidad 1]
    degree=0.0753  betweenness=0.0682  eigenvector=0.2729
    Justificación: alto volumen de interacción directa, actú

,node,tipo,equipo,comunidad,degree_centrality,betweenness_centrality,eigenvector_centrality,justificacion
0,equipo_usa,seleccion,usa,1,0.1720,0.2517,0.3827,alto volumen de interacción directa; actúa com...
1,equipo_mexico,seleccion,mexico,5,0.1935,0.3077,0.3798,alto volumen de interacción directa; actúa com...
2,equipo_canada,seleccion,canada,2,0.1828,0.3325,0.3609,alto volumen de interacción directa; actúa com...
3,bbc_sports,medio,global,1,0.0753,0.0682,0.2729,alto volumen de interacción directa; actúa com...
4,fifa_news,medio,global,2,0.0645,0.2608,0.2536,alto volumen de interacción directa; actúa com...
5,influencer_futbol_03,influencer,global,1,0.0430,0.0242,0.2209,alto volumen de interacción directa; conectada...
6,bot_fut_01,bot,global,2,0.0430,0.0242,0.2179,alto volumen de interacción directa; conectada...
7,influencer_futbol_01,influencer,global,2,0.0430,0.0242,0.2179,alto volumen de interacción directa; conectada...
8,fox_sports_global,medio,global,5,0.0538,0.0457,0.2167,alto volumen de interacción directa; conectada...
9,influencer_futbol_02,influencer,global,5,0.0430,0.0242,0.2120,alto volumen de interacción directa; conectada...


## 4. Detección de bots y su impacto en la conversación

**Objetivo:** detectar bots y analizar cómo alteran la estructura de la conversación.

También se apoya en el módulo externo `centralidad.py`:

- `detectar_bots(G, centralidad_df, nodes_df, particion)`: a partir del grafo, las
  centralidades (Sección 3), los metadatos de cuentas y la partición de comunidades
  (Sección 2), identifica cuentas con comportamiento compatible con un bot (p. ej.
  patrones de mención atípicos, concentración de actividad, etiqueta `tipo == "bot"`
  en `nodes.csv`, combinada con métricas estructurales). Devuelve:
  - `bots_confirmados`: bots identificados con alta confianza.
  - `candidatos_revision`: cuentas sospechosas que requieren revisión manual.
  - `bots_detalle_df`: `DataFrame` con el detalle completo de la detección.

Esta sección conecta directamente con la Sección 2 (comunidades con alto `pct_bots`
son señaladas como posible amplificación artificial) y con la Sección 3 (bots que
alcanzan alta centralidad son especialmente relevantes para el análisis de
desinformación).


In [ ]:
bots_confirmados, candidatos_revision, bots_detalle_df = detectar_bots(
    G, centralidad_df, nodes_df, particion
)
bots_detalle_df

TAREA 4 — DETECCIÓN DE BOTS

Bots confirmados (lista conocida, nodes.csv): 5
  - bot_fut_01
  - bot_fut_02
  - bot_fut_03
  - bot_fut_04
  - bot_fut_05

Candidatos adicionales por regla suave (revisión manual, NO confirmados): 3
  - fan_can_10  (posible cuenta amplificada por un bot, no necesariamente un bot)
  - fan_mex_10  (posible cuenta amplificada por un bot, no necesariamente un bot)
  - fan_usa_10  (posible cuenta amplificada por un bot, no necesariamente un bot)

--- bot_fut_01 ---
Comunidad: 2
Menciona a: seleccion_mexico (peso 1), seleccion_usa (peso 1), seleccion_canada (peso 1), fifa_news (peso 1)
Es mencionado por: (nadie)

--- bot_fut_02 ---
Comunidad: 1
Menciona a: seleccion_mexico (peso 1), espn_usa (peso 1), cbc_sports_01 (peso 1)
Es mencionado por: (nadie)

--- bot_fut_03 ---
Comunidad: 6
Menciona a: fan_mex_05 (peso 1), fan_usa_06 (peso 1), fan_can_07 (peso 1)
Es mencionado por: (nadie)

--- bot_fut_04 ---
Comunidad: 4
Menciona a: fan_mex_10 (peso 1), fan_usa_10 (pes

,bot,comunidad,menciona_a,mencionado_por
0,bot_fut_01,2,equipo_mexico (peso 1); equipo_usa (peso...,
1,bot_fut_02,1,equipo_mexico (peso 1); espn_usa (peso 1); ...,
2,bot_fut_03,6,fan_mex_05 (peso 1); fan_usa_06 (peso 1); fan_...,
3,bot_fut_04,4,fan_mex_10 (peso 1); fan_usa_10 (peso 1); fan_...,
4,bot_fut_05,5,periodista_mx_01 (peso 1); periodista_usa_01 (...,


## 5. Dashboard / reporte visual de hallazgos

**Objetivo:** presentar los hallazgos en un dashboard o reporte visual.

Función importada desde `centralidad.py`:

- `generar_visualizacion(G, particion, centralidad_df, nodes_df, bots_confirmados, modularidad, output_path=...)`:
  genera una visualización interactiva (HTML) del grafo, coloreando por comunidad
  (Sección 2), dimensionando por centralidad (Sección 3) y resaltando los bots
  detectados (Sección 4). Incluye la modularidad global como contexto del análisis.
  El resultado se guarda en la ruta indicada por `output_path`
  (por defecto `grafo_nfl2014.html` en el script original).


In [ ]:
generar_visualizacion(
    G, particion, centralidad_df, nodes_df, bots_confirmados, modularidad,
    output_path="grafo_nfl2014.html",
)

TAREA 5 — VISUALIZACIÓN INTERACTIVA (PyVis)

HTML interactivo generado en: grafo_nfl2014.html


## 6. Código main

**Objetivo:** orquestar de punta a punta todas las secciones anteriores (1 a 5),
tal como lo hace el script original al ejecutarse desde línea de comandos:

```
python nfl2014_analisis_red.py
python nfl2014_analisis_red.py --edges edges.csv --nodes nodes.csv
```

El `main()` original:

1. Parsea argumentos CLI (`--edges`, `--nodes`, `--out`, `--out-centralidad`,
   `--out-bots`, `--out-html`).
2. Carga los datos (`cargar_datos`) → **Sección 1** (datos de entrada).
3. Construye el grafo y reporta estadísticas → **Sección 1**.
4. Detecta comunidades e interpreta cada una → **Sección 2**, y guarda
   `resumen_comunidades_nfl.csv`.
5. Calcula centralidades y el top de influencers → **Sección 3**, y guarda
   `centralidad_nfl.csv`.
6. Detecta bots → **Sección 4**, y guarda `bots_detectados_nfl.csv`.
7. Genera la visualización final → **Sección 5**, y guarda `grafo_nfl2014.html`.

El código se reproduce exactamente igual al original; en un notebook normalmente
no se invoca `main()` directamente con `argparse` (porque no hay línea de comandos
real), pero se deja documentado para ejecutarlo como script (`.py`) o para
llamarlo manualmente pasando rutas por defecto.


In [ ]:

parser = argparse.ArgumentParser(
    description="Análisis de red social sobre la NFL 2014 (grafo + comunidades Louvain)."
)
parser.add_argument("--edges", default="edges.csv", help="Ruta al archivo edges.csv")
parser.add_argument("--nodes", default="nodes.csv", help="Ruta al archivo nodes.csv")
parser.add_argument(
    "--out",
    default="resumen_comunidades_nfl.csv",
    help="Ruta de salida para el resumen de comunidades",
)
parser.add_argument(
    "--out-centralidad",
    default="centralidad_nfl.csv",
    help="Ruta de salida para el top de centralidad/influencers",
)
parser.add_argument(
    "--out-bots",
    default="bots_detectados_nfl.csv",
    help="Ruta de salida para el detalle de bots (Tarea 4)",
)
parser.add_argument(
    "--out-html",
    default="grafo_nfl2014.html",
    help="Ruta de salida para la visualización interactiva (Tarea 5)",
)
args = parser.parse_args(args=["--edges", "edges.csv", "--nodes", "nodes.csv"])

edges_df, nodes_df = cargar_datos(args.edges, args.nodes)

G = construir_grafo(edges_df)
reportar_grafo(G)

particion, comunidades, modularidad = detectar_comunidades(G)
resumen_df = interpretar_comunidades(comunidades, nodes_df)

resumen_df.to_csv(args.out, index=False)
print(f"\nResumen de comunidades guardado en: {args.out}")

centralidad_df = calcular_centralidades(G)
top_influencers_df = reportar_centralidades(centralidad_df, nodes_df, particion, top_n=10)

top_influencers_df.to_csv(args.out_centralidad, index=False)
print(f"Top de influencers guardado en: {args.out_centralidad}")

bots_confirmados, candidatos_revision, bots_detalle_df = detectar_bots(
    G, centralidad_df, nodes_df, particion
)
bots_detalle_df.to_csv(args.out_bots, index=False)
print(f"Detalle de bots guardado en: {args.out_bots}")

generar_visualizacion(
    G, particion, centralidad_df, nodes_df, bots_confirmados, modularidad,
    output_path=args.out_html,
)

TAREA 1 — CONSTRUCCIÓN DEL GRAFO
Número de nodos : 94
Número de enlaces (aristas): 132
Densidad del grafo: 0.037386

Top 5 cuentas por grado ponderado (volumen de interacción):
  seleccion_mexico: 18
  seleccion_canada: 17
  seleccion_usa: 16
  espn_usa: 7
  bbc_sports: 7

TAREA 2 — DETECCIÓN DE COMUNIDADES (LOUVAIN)
Modularidad global: 0.5442
Número de comunidades detectadas: 8

Tamaño de cada comunidad:
  Comunidad 1: 11 nodos
  Comunidad 3: 10 nodos
  Comunidad 9: 2 nodos
  Comunidad 2: 7 nodos
  Comunidad 6: 6 nodos
  Comunidad 7: 4 nodos
  Comunidad 8: 2 nodos
  Comunidad 9: 2 nodos

INTERPRETACIÓN DE COMUNIDADES

--- Comunidad 5 (16 nodos) ---
País dominante: mexico (68.8% de la comunidad)
Distribución por país:
    mexico: 11
    global: 4
    canada: 1
Distribución por tipo de cuenta:
    periodista: 5
    fan: 4
    medio: 3
    seleccion: 1
    influencer: 1
    bot: 1
    hashtag: 1
Interpretación: Comunidad de medios/cobertura global (conecta varios países)

--- Comunidad 1